## 🏆 Best Performing Notebook — NeuralHive Kaggle Challenge 2025

This notebook represents the **top-performing solution** from the *NeuralHive Kaggle Challenge 2025*.  
The solution was developed by **Chatresh Ramasai Gudi**, showcasing an effective approach, methodology, and model pipeline that achieved the highest leaderboard score.# 🏆 **Best Performing Notebook — NeuralHive Kaggle Challenge 2025**

### **Author:** *Chatresh Ramasai Gudi*

This notebook represents the **top-performing solution** submitted for the  
**NeuralHive Kaggle Challenge 2025**.

It showcases:
- A well-structured approach  
- An optimized modeling pipeline  
- Techniques that achieved the **highest score on the leaderboard**

Explore the methodology, insights, and implementation that set this solution apart.

## 1. Feature Selection

This section outlines the complete set of predictor variables used for model training.  
The feature list includes both **raw attributes from the original dataset** and **engineered features** introduced to enhance the model’s ability to learn complex relationships.

### **Original Dataset Features**
These variables are directly supplied in the provided training data and represent the core agronomic, temporal, and environmental attributes:

- `State_Name` — Categorical geographic identifier  
- `District_Name` — Finer-grained location information  
- `Crop_Year` — Temporal indicator capturing year-specific patterns  
- `Season` — Seasonal categorization for crop cycles  
- `Crop` — Crop type, driving both yield characteristics and input dependencies  
- `Area` — Cultivated land area (a primary determinant of production)  
- `Rainfall_mm` — Seasonal rainfall; proxy for water availability  
- `Temperature_C` — Seasonal temperature; affects crop growth and stress tolerance

*(Note: `Production` is the target variable and is therefore excluded from `feature_cols`.)*

### **Engineered Features**
The following features do not exist in the original dataset and were introduced by the participant.  
They represent **domain-inspired and synthetic signals** designed to increase model expressiveness:

- `Fertilizer_kg_per_ha` — Approximates input intensity; correlates with yield potential  
- `Soil_Quality` — Captures soil fertility characteristics  
- `Irrigation_Index` — Indicates water accessibility beyond rainfall  
- `Govt_Subsidy_Score` — Models policy-driven support affecting agricultural output  
- `Market_Accessibility` — Reflects ease of distribution, influencing crop choice and yield  
- `Random_Noise_Factor` — A synthetic perturbation variable that may help reduce overfitting by adding stochastic variation

These engineered features expand the feature space, enabling the model to learn more nuanced patterns and improve predictive performance.

In [ ]:
import pandas as pd

# Select common features
feature_cols = [
    'State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area',
    'Rainfall_mm', 'Temperature_C', 'Fertilizer_kg_per_ha', 'Soil_Quality',
    'Irrigation_Index', 'Govt_Subsidy_Score', 'Market_Accessibility',
    'Random_Noise_Factor'
]

## 2. Data Loading and Matrix Construction

With the feature set defined, the next step is to load the dataset and organize it into the components required for supervised learning.

### **Loading Input Files**
The training and test CSV files are read using pandas, allowing efficient ingestion of structured tabular data. This provides two DataFrames (`train` and `test`) that serve as the foundation for feature extraction and model preparation.

### **Constructing Feature Matrices**
From these DataFrames, the predefined feature columns (`feature_cols`) are selected to form the input matrices:

- **X** — the feature matrix used for model training  
- **X_test** — the feature matrix used for generating predictions on unseen data  

Each selection is copied to ensure that downstream transformations do not unintentionally mutate the original DataFrame slices.

### **Extracting Targets and Identifiers**
Two additional components are prepared for the ML pipeline:

- **y** — the target vector containing the ground-truth `Production` values from the training dataset  
- **test_ids** — the unique row identifiers from the test dataset, required for assembling the final Kaggle submission file  

This separation of features, targets, and metadata establishes a clean and reproducible foundation for the modeling workflow, ensuring clarity and modularity throughout subsequent steps.

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

X = train[feature_cols].copy()
X_test = test[feature_cols].copy()

y = train['Production'].copy()
test_ids = test['Id']

## 3. Encoding Categorical Features

Many of the selected input features are categorical (e.g., state names, districts, crop types, and seasons). Machine learning models—especially tree-based or linear algorithms—cannot directly interpret string-based data. Therefore, these variables must be transformed into numerical form.

### **Identifying Text-Based Columns**
The feature matrix is inspected to automatically detect all columns with `object` datatype, ensuring that every categorical variable is captured without manual enumeration.

### **Consistent Encoding Across Train and Test Sets**
A **LabelEncoder** is applied to each categorical column. To avoid mismatched label mappings between the training and test datasets, the encoder is fitted on the *combined* values from both sets. This ensures:

- All categories seen in either dataset are accounted for  
- No category in the test set will be encoded incorrectly or left unseen  

### **Transforming the Data**
After fitting, the encoder transforms both `X` and `X_test`, replacing each categorical value with a stable, integer-based representation. This step standardizes all non-numeric attributes, making the feature matrices fully compatible with downstream ML algorithms.

This careful, synchronized encoding process preserves information while ensuring model stability during training and inference.

In [3]:
from sklearn.preprocessing import LabelEncoder

# Identify columns with string values
text_cols = X.select_dtypes(include=['object']).columns.tolist()

for col in text_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col].astype(str), X_test[col].astype(str)])
    le.fit(combined)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))


## 4. Model Selection and Training (ExtraTrees Regressor)

With the dataset fully prepared, the next phase involves selecting a robust learning algorithm capable of handling a heterogeneous feature set and capturing complex nonlinear relationships within the data.  
For this solution, an **ExtraTreesRegressor**—a highly efficient ensemble-based model—is employed.

### **Why ExtraTreesRegressor?**
ExtraTrees (Extremely Randomized Trees) is an ensemble method that constructs multiple decision trees and averages their predictions. It is well-suited for this task because:

- It handles both numerical and encoded categorical features effectively  
- It reduces variance through ensemble averaging  
- It introduces extra randomness in tree splits, often improving generalization  
- It is computationally efficient even with large feature spaces  

### **Model Configuration**
The regressor is initialized with carefully chosen hyperparameters:

- **n_estimators = 300** — a large number of trees for stable predictions  
- **max_depth = 20** — controls model complexity and prevents overfitting  
- **min_samples_split = 10** — avoids overly granular splits  
- **random_state = 42** — ensures reproducible results  
- **n_jobs = 1** — forces single-thread execution for consistent runtime behavior  

These settings aim to balance model expressiveness with computational efficiency.

### **Training and Prediction**
The model is trained using the full training matrix **X** and target vector **y**, allowing the ensemble to learn production patterns from both numerical and encoded categorical features.  
Once trained, the regressor generates predictions on **X_test**, producing the final output values that will populate the submission file.

This step forms the core learning component of the pipeline, translating engineered features into accurate production forecasts.

In [4]:
from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor(
    n_estimators=300,         # Number of trees
    max_depth=20,             # Maximum depth
    min_samples_split=10,     # Minimum samples to split
    random_state=42,          # For reproducible results
    n_jobs=1                  # Single thread
)
et.fit(X, y)
predictions = et.predict(X_test)


## 5. Generating the Submission File

With the model’s predictions ready, the final step is to structure the output in the exact format required by the Kaggle competition and export it as a CSV file.

### **Post-processing Predictions**
Since the target variable represents **crop production**, it cannot logically take negative values.  
To enforce this constraint, all predictions are passed through `np.maximum(predictions, 0)`, ensuring that any negative outputs are clipped to zero.  
This maintains physical realism and avoids penalization during evaluation.

### **Assembling the Submission DataFrame**
A submission DataFrame is created with two essential components:

- **Id** — the unique identifier from the test dataset  
- **Expected** — the model’s final predicted production values  

This structure aligns with the competition’s required submission schema.

### **Exporting the Submission**
The DataFrame is written to a CSV file (`extratrees_only.csv`) with index suppression for compatibility with Kaggle’s upload format.  
A confirmation message is printed to indicate successful file generation.

This step finalizes the predictive pipeline, producing a clean, competition-ready submission file derived directly from the trained model.

In [5]:
import numpy as np
submission = pd.DataFrame({
    'Id': test_ids,
    'Expected': np.maximum(predictions, 0)
})
submission.to_csv('extratrees_only.csv', index=False)
print('Submission saved: extratrees_only.csv')

Submission saved: extratrees_only.csv
